# Análisis Descriptivo de Micronegocios en Cesar
**Fuente:** DANE – Encuesta de Micronegocios (EMICRON) – Módulo características del micronegocio
**Departamento:** Cesar (COD_DEPTO = 20)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False, 'font.size': 11})

RUTA = (r'C:\Users\Usuario\OneDrive - Global Green Growth Institute'
        r'\Documentos\2025\Adapta Cesar\micronegocios_cesar.csv')
df_raw = pd.read_csv(RUTA, encoding='latin-1', low_memory=False)
print(f'Filas totales: {len(df_raw):,}  |  Columnas: {df_raw.shape[1]}')
cesar = df_raw[df_raw['COD_DEPTO'] == 20].copy()
universo = cesar['F_EXP'].sum()
print(f'Registros muestra Cesar: {len(cesar):,}')
print(f'Universo estimado de micronegocios en Cesar: {universo:,.0f}')

## 1. Diccionario de variables (Módulo características del micronegocio)

In [ ]:
diccionario = {
    'P1633':    'Registro en RUT (1=Sí, 2=No)',
    'P986':     'Régimen tributario (1=Simplificado, 2=Común/ordinario, 9=No sabe)',
    'P640':     'Lleva registros contables (1=Sí, 2=No)',
    'P4000':    'Razón principal para no llevar registro contable (1-8)',
    'P1055':    'Matriculado en Cámara de Comercio (1=Sí, 2=No)',
    'P1056':    'Año de matrícula en Cámara de Comercio',
    'P661':     'Matrícula mercantil renovada (1=Sí, 2=No)',
    'P1057':    'Razón principal para no estar matriculado (1-8)',
    'P4004':    'Año de vencimiento de la matrícula',
    'P2991':    'Presentó declaración de renta último año (1=Sí, 2=No)',
    'P2992':    'Presentó declaración de IVA último año (1=Sí, 2=No)',
    'P2993':    'Presentó declaración de ICA último año (1=Sí, 2=No)',
    'CLASE_TE': 'Tipo establecimiento (1=Vivienda,2=Local,3=Vía pública,4=Obra,5=Vehículo,6=Otro)',
    'AREA':     'Área geográfica (1=Cabecera, 2=Centros poblados, 3=Rural disperso)',
    'F_EXP':    'Factor de expansión (peso muestral)'
}
pd.DataFrame.from_dict(diccionario, orient='index', columns=['Descripción']).rename_axis('Variable')

## 2. Distribución por zona geográfica (AREA)

In [ ]:
zona_map = {1: 'Cabecera', 2: 'Centros poblados', 3: 'Rural disperso'}
zona = cesar.groupby('AREA')['F_EXP'].sum().rename(index=zona_map)
zona_pct = (zona / zona.sum() * 100).round(1)

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.barh(zona.index, zona.values, color=['#2196F3', '#4CAF50', '#FF9800'])
ax.bar_label(bars, labels=[f'{v:,.0f} ({p}%)' for v, p in zip(zona.values, zona_pct.values)], padding=5)
ax.set_xlabel('Micronegocios estimados')
ax.set_title('Micronegocios por zona geográfica – Cesar')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.tight_layout(); plt.show()
print(zona.to_frame('Universo').assign(Pct=zona_pct))

## 3. Formalización tributaria – RUT (P1633) y Régimen (P986)

In [ ]:
rut_map = {1: 'Sí tiene RUT', 2: 'No tiene RUT'}
reg_map = {1: 'Simplificado', 2: 'Común/Ordinario', 9: 'No sabe'}
rut = cesar.groupby('P1633')['F_EXP'].sum().rename(index=rut_map)
reg = cesar[cesar['P1633'] == 1].groupby('P986')['F_EXP'].sum().rename(index=reg_map)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].pie(rut.values, labels=rut.index, autopct='%1.1f%%',
            colors=['#4CAF50', '#F44336'], startangle=90)
axes[0].set_title('Registro en RUT (P1633)')
axes[1].pie(reg.values, labels=reg.index, autopct='%1.1f%%',
            colors=['#2196F3', '#FF9800', '#9E9E9E'], startangle=90)
axes[1].set_title('Régimen tributario con RUT (P986)')
plt.suptitle('Formalización tributaria – Micronegocios Cesar', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()
print('RUT:', rut.to_frame('Universo').assign(Pct=(rut/rut.sum()*100).round(1)))
print('\nRégimen (con RUT):', reg.to_frame('Universo').assign(Pct=(reg/reg.sum()*100).round(1)))

## 4. Registros contables (P640) y razón para no llevarlos (P4000)

In [ ]:
cont_map = {1: 'Lleva registros', 2: 'No lleva registros'}
razon_cont_map = {1:'No lo considera necesario',2:'No sabe cómo',3:'No tiene tiempo',
                  4:'Es muy costoso',5:'No lo exigen',6:'Negocio muy pequeño',
                  7:'Lo hace mentalmente',8:'Otro'}
cont = cesar.groupby('P640')['F_EXP'].sum().rename(index=cont_map)
razon_no = cesar[cesar['P640'] == 2].groupby('P4000')['F_EXP'].sum().rename(index=razon_cont_map)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].pie(cont.values, labels=cont.index, autopct='%1.1f%%',
            colors=['#4CAF50', '#F44336'], startangle=90)
axes[0].set_title('¿Lleva registros contables? (P640)')
rs = razon_no.sort_values(ascending=True)
axes[1].barh(rs.index, rs.values, color='#FF9800')
axes[1].bar_label(axes[1].containers[0], labels=[f'{v:,.0f}' for v in rs.values], padding=4)
axes[1].set_xlabel('Micronegocios estimados')
axes[1].set_title('Razón para no llevar contabilidad (P4000)')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.suptitle('Registros contables – Micronegocios Cesar', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

## 5. Cámara de Comercio (P1055, P661, P1057)

In [ ]:
camara_map = {1: 'Matriculado', 2: 'No matriculado'}
renov_map  = {1: 'Renovada', 2: 'No renovada'}
razon_cam_map = {1:'No lo considera necesario',2:'Es muy costoso',3:'No sabe cómo',
                 4:'No lo exigen',5:'Trámite difícil',6:'No tiene tiempo',
                 7:'Negocio muy pequeño',8:'Otro'}
camara = cesar.groupby('P1055')['F_EXP'].sum().rename(index=camara_map)
renov  = cesar[cesar['P1055'] == 1].groupby('P661')['F_EXP'].sum().rename(index=renov_map)
razon_cam = cesar[cesar['P1055'] == 2].groupby('P1057')['F_EXP'].sum().rename(index=razon_cam_map)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].pie(camara.values, labels=camara.index, autopct='%1.1f%%',
            colors=['#4CAF50', '#F44336'], startangle=90)
axes[0].set_title('Matrícula Cámara de Comercio (P1055)')
axes[1].pie(renov.values, labels=renov.index, autopct='%1.1f%%',
            colors=['#2196F3', '#FF5722'], startangle=90)
axes[1].set_title('Matrícula renovada (P661)')
rcs = razon_cam.sort_values(ascending=True)
axes[2].barh(rcs.index, rcs.values, color='#9C27B0')
axes[2].bar_label(axes[2].containers[0], labels=[f'{v:,.0f}' for v in rcs.values], padding=4)
axes[2].set_xlabel('Micronegocios estimados')
axes[2].set_title('Razón para no matricularse (P1057)')
axes[2].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.suptitle('Cámara de Comercio – Micronegocios Cesar', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

## 6. Declaraciones tributarias: Renta, IVA e ICA (P2991–P2993)

In [ ]:
tribut_vars = {'P2991': 'Renta', 'P2992': 'IVA', 'P2993': 'ICA'}
resumen = []
for var, etiqueta in tribut_vars.items():
    si  = cesar[cesar[var] == 1]['F_EXP'].sum()
    no  = cesar[cesar[var] == 2]['F_EXP'].sum()
    tot = si + no
    resumen.append({'Declaración': etiqueta, 'Sí presentó': si, 'No presentó': no, '% Sí': round(si/tot*100,1)})
df_trib = pd.DataFrame(resumen).set_index('Declaración')
print(df_trib)

fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(df_trib)); w = 0.35
b1 = ax.bar(x - w/2, df_trib['Sí presentó'], w, label='Sí presentó', color='#4CAF50')
b2 = ax.bar(x + w/2, df_trib['No presentó'], w, label='No presentó', color='#F44336')
ax.set_xticks(x); ax.set_xticklabels(df_trib.index)
ax.set_ylabel('Micronegocios estimados')
ax.set_title('Declaraciones tributarias – Micronegocios Cesar')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:,.0f}'))
ax.bar_label(b1, labels=[f'{v:,.0f}' for v in df_trib['Sí presentó']], rotation=45, padding=3, fontsize=9)
ax.bar_label(b2, labels=[f'{v:,.0f}' for v in df_trib['No presentó']], rotation=45, padding=3, fontsize=9)
ax.legend(); plt.tight_layout(); plt.show()

## 7. Índice sintético de formalización (3 dimensiones)

In [ ]:
cesar['formal_rut']    = (cesar['P1633'] == 1).astype(int)
cesar['formal_cont']   = (cesar['P640']  == 1).astype(int)
cesar['formal_camara'] = (cesar['P1055'] == 1).astype(int)
cesar['indice_formal'] = cesar[['formal_rut','formal_cont','formal_camara']].sum(axis=1)

idx_dist = cesar.groupby('indice_formal')['F_EXP'].sum()
idx_pct  = (idx_dist / idx_dist.sum() * 100).round(1)
etiq = ['0 – Sin formalización','1 – Una dimensión','2 – Dos dimensiones','3 – Completamente formal']
cols = ['#F44336','#FF9800','#2196F3','#4CAF50']

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(etiq[:len(idx_dist)], idx_dist.values, color=cols[:len(idx_dist)])
ax.bar_label(bars, labels=[f'{v:,.0f}\n({p}%)' for v, p in zip(idx_dist.values, idx_pct.values)], padding=4)
ax.set_ylabel('Micronegocios estimados')
ax.set_title('Índice sintético de formalización (0–3) – Cesar')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:,.0f}'))
plt.xticks(rotation=15, ha='right'); plt.tight_layout(); plt.show()

def tasa(df, var):
    return df[df[var] == 1]['F_EXP'].sum() / df['F_EXP'].sum() * 100

print('Tasa de formalización por dimensión:')
for var, nombre in [('formal_rut','RUT'),('formal_cont','Contabilidad'),('formal_camara','Cámara de Comercio')]:
    print(f'  {nombre}: {tasa(cesar, var):.1f}%')

## 8. Tipo de establecimiento (CLASE_TE)

In [ ]:
clase_map = {1:'Vivienda/parte de vivienda',2:'Local/oficina/bodega',
             3:'Vía pública/espacio abierto',4:'En obra/construcción',5:'Vehículo',6:'Otro'}
clase = cesar.groupby('CLASE_TE')['F_EXP'].sum().rename(index=clase_map).sort_values(ascending=False)
clase_pct = (clase / clase.sum() * 100).round(1)

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(clase.index[::-1], clase.values[::-1], color='#00897B')
ax.bar_label(bars, labels=[f'{v:,.0f} ({p}%)'
             for v, p in zip(clase.values[::-1], clase_pct.values[::-1])], padding=5)
ax.set_xlabel('Micronegocios estimados')
ax.set_title('Tipo de establecimiento – Micronegocios Cesar (CLASE_TE)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:,.0f}'))
plt.tight_layout(); plt.show()

## 9. Comparación Cesar vs. total nacional

In [ ]:
nacional = df_raw.copy()
nacional['formal_rut']    = (nacional['P1633'] == 1).astype(int)
nacional['formal_cont']   = (nacional['P640']  == 1).astype(int)
nacional['formal_camara'] = (nacional['P1055'] == 1).astype(int)

comparacion = pd.DataFrame({
    'Cesar':    [tasa(cesar,   'formal_rut'), tasa(cesar,   'formal_cont'), tasa(cesar,   'formal_camara')],
    'Nacional': [tasa(nacional,'formal_rut'), tasa(nacional,'formal_cont'), tasa(nacional,'formal_camara')]
}, index=['RUT','Registros contables','Cámara de Comercio']).round(1)
print(comparacion)

fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(len(comparacion)); w = 0.35
b1 = ax.bar(x - w/2, comparacion['Cesar'],    w, label='Cesar',    color='#2196F3')
b2 = ax.bar(x + w/2, comparacion['Nacional'], w, label='Nacional', color='#9E9E9E')
ax.set_xticks(x); ax.set_xticklabels(comparacion.index)
ax.set_ylabel('Tasa de formalización (%)')
ax.set_title('Formalización: Cesar vs. Nacional')
ax.bar_label(b1, labels=[f'{v:.1f}%' for v in comparacion['Cesar']],    padding=3)
ax.bar_label(b2, labels=[f'{v:.1f}%' for v in comparacion['Nacional']], padding=3)
ax.set_ylim(0, 100); ax.legend(); plt.tight_layout(); plt.show()

## 10. Resumen ejecutivo

In [ ]:
univ       = cesar['F_EXP'].sum()
pct_rut    = tasa(cesar, 'formal_rut')
pct_cont   = tasa(cesar, 'formal_cont')
pct_camara = tasa(cesar, 'formal_camara')
pct_3dim   = cesar[cesar['indice_formal'] == 3]['F_EXP'].sum() / univ * 100
pct_0dim   = cesar[cesar['indice_formal'] == 0]['F_EXP'].sum() / univ * 100
pct_renta  = (cesar[cesar['P2991'] == 1]['F_EXP'].sum()
              / cesar[cesar['P2991'].isin([1,2])]['F_EXP'].sum() * 100)

print('=' * 62)
print('RESUMEN EJECUTIVO – MICRONEGOCIOS CESAR (EMICRON DANE)')
print('=' * 62)
print(f'Universo estimado de micronegocios:  {univ:>12,.0f}')
print('-' * 62)
print('DIMENSIÓN TRIBUTARIA')
print(f'  Con RUT registrado:                {pct_rut:>11.1f}%')
print(f'  Declararon renta (último año):     {pct_renta:>11.1f}%')
print('-' * 62)
print('DIMENSIÓN CONTABLE')
print(f'  Llevan registros contables:        {pct_cont:>11.1f}%')
print('-' * 62)
print('DIMENSIÓN LEGAL')
print(f'  Matriculados Cámara de Comercio:   {pct_camara:>11.1f}%')
print('-' * 62)
print('ÍNDICE SINTÉTICO (3 dimensiones)')
print(f'  Completamente formales (3/3):      {pct_3dim:>11.1f}%')
print(f'  Completamente informales (0/3):    {pct_0dim:>11.1f}%')
print('=' * 62)